<a href="https://colab.research.google.com/github/RileyMarek/NasaRoverVLModel/blob/main/Marek_AutonomousRoverSystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NASA Rover Research Project**
****
**ML Model: Vision-Language segmentaion using U-Net Architecture**

Student: Riley Marek

WSUID: 11843302

Project Proposal:

https://emailwsu-my.sharepoint.com/:w:/g/personal/riley_marek_wsu_edu/IQBLEOWgFzYdSo3YjrjukNyjAarehlioB3FybL-p5Mr0PpE?e=OBSQDD





In [ ]:
# Install necessary libraries
!pip install -q segmentation-models-pytorch albumentations transformers torchmetrics

**Data Processing:**
****

In [ ]:
import cv2
import numpy as np
import albumentations as A
from ablumentations.pytorch import ToTensorV2
import torch
from torch.utils.data import Dataset, DataLoader

class LunarTerrainDataset(Dataset):
  def __init__(self, image_paths, mask_paths, prompts, transform=None):
    self.image_paths = image_paths
    self.mask_paths = mask_paths
    self.prompts = prompts
    self.transform = transform

    # Initialize CLAHE for lunar contrast normalization
    self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
  def __len__self(self):
    return len(self.image_paths)

  def __getitem__(self, idx):

    # 1. load image and mask
    image = cv2.imread(self.image_paths[idx])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)


    # 2. CLAHE pre-proccessing (improve local contrast and enhance edge definition)
    image_bw = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    image = self.clahe.apply(image_bw)
    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

    # 3. Apply Albumentations (Shadows, Noise, etc)
    if self.transform:
      augmented = self.transform(image=image, mask=mask)
      image = augmented['image']
      mask = augmented['mask']

    # 4. Handle language prompt
    prompt = self.prompts[idx]

    return image, mask, prompt


# -- Define Albumentation Strategy --
train_transform = A.Compose(
    [
        A.Resize(height=512, width=512),

        # Simulate Lunar Shadows (for lunar nav)
        A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), num_shadows_lower=1,num_shadows_upper=2, shadow_dimension=5, p=0.5),

        # Add Gaussian Noise to simulate sensor limitations
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)


**FiLM Layer:**
****

In [ ]:
import torch
import torch.nn as nn

class FiLMLayer(nn.Module):
  def __init__(self, text_emb_dim, feature_channels):
    super().__init__()
    # Generate gamma (scale) and beta (shift) from text embedding
    self.generator = nn.Linear(text_emb_dim, 2 * feature_channels)

  def forward(self, x, text_embedding):
    # x shape: [Batch, Channels, H, W]
    # text_embedding shape: [Batch, text_emb_dim]

    # Predict parameters
    params = self.generator(text_embedding)
    params = params.unsqueeze(-1).unsqueeze(-1) # [Batch, 2*feature_channels, 1, 1]
    gamma, beta = torch.chunk(params, chunks=2, dim=1)

    # Apply Afine transformation: (gamma * x) + beta
    return gamma * x + beta


**Multi-Modal (Vision-Language) U-Net Architecture:**
****

In [ ]:
from transformers import CLIPVisionModel, CLIPTextModel, CLIPProccessor

class VLUNet(nn.Module):
  def __init__(self, num_classes=1):
    super.__init__()

    # Frozen encoders (might fine-tune later depending on compute req)
    self.vision_encoder = CLIPVisionModel.from_pretrained('openai/clip-vit-base-patch32')
    self.text_encoder = CLIPTextModel.from_pretrained('openai/clip-vit-base-patch32')

    for param in self.vision_encoder.parameters(): param.requires_grad = False
    for param in self.text_encoder.parameters(): param.requires_grad = False

    # FiLM conditioning at Bottleneck
    # CLIP-base-patch32 output hidden size is 768 for vision
    self.bottleneck_film = FiLMlayer(text_emb_dim=512, feature_channels=768)

    # Decoder (U-Net for now, might switch to SMP decoder)
    #self.upsample = nn.ConvTranspose2d(768, 256, kernel_size=2, stride=2)
    #self.final_conv = nn.Conv2d(256, num_classes, kernel_size=1)

    # SMP decoder
    self.decoder = UnetDecoder(
        encoder_channels=[768, 768, 768, 768, 768],
        decoder_channels=[256, 128, 64, 32, 16],
        n_blocks=5,
    )

    self.segmentation_head = nn.Conv2d(16, num_classes, kernel_size=1)

  def forward(self, pixel_values, input_ids, attention_mask):

    # Get language embedding
    text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
    text_emb = text_outputs.pooler_output # [Batch, 512]

    # Get vision features
    vision_outputs = self.vision_encoder(pixel_values=pixel_values, output_hidden_states=True)

    # Select Layers
    raw_layers = [vision_outputs.hidden_states[i] for i in [3, 6, 9, 11, 12]
    # CLIP ViT outputs patches; we reshape them back to a 2D grid
    # For ViT-B/32 on 224x224, that's 7x7 patches
    last_hidden_state = vision_outputs.last_hidden_states[:, 1:, :] # This removes CLS token
    grid_size = int(last_hidden_state.shape[1] ** 0.5)
    visual_features = last_hidden_state.transpose(1, 2).view(-1, 768, grid_size, grid_size)

    # Apply FiLM at bottleneck
    conditioned_features = self.bottleneck_film(visual_features, text_emb)

    # Decoder/Upsampling
    x = self.upsample(conditioned_features)


    # Add in skip connections (to-do later)


    return self.final_conv(x)

**Hybrid Loss Function**
****
Binary Cross-Entropy for pixel-wise confidence

Dice-Loss for regional overlap

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceBCELoss(nn.Module):
  def __init__(self, weight=None, size_average=True):
    super(DiceBCELoss, self).__init__()

  def forward(self, inputs, targets, smooth=1):

    # flatten label and prediction tensors
    inputs = torch.sigmoids(inputs)
    inputs = inputs.view(-1)
    targets = targets.view(-1)

    # Dice Loss
    intersection = (inputs * targets).sum()
    dice_loss = 1 - (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)

    # Binary Cross-Entropy
    BCE = F.binary_cross_entropy(inputs, targets, reduction='mean')

    # Combine
    return BCE + dice_loss


**Pull LRO Data**
****

In [ ]:
import os
import requests

def download_lro_subset(image_ids, save_dir="lro_data"):
  os.makedirs(save_dir, exist_ok=True)
  base_url = "https://wms.lroc.asu.edu/lroc/download_product/"

  for image_id in image_ids:
    url = "f{base_url}{image_id}/EDR"
    print(f"Fetching {img_id}...")
    r = requests.get(url, stream=True)
    if r.status_code = 200:
      with open(f"{save_dir}/{image_id}.IMG", "wb") as f:
        for chunk in r.iter_content(chunk_size=1024):
          f.write(chunk)
    else:
      print(f"Failed to fetch {img_id}")


# Example IDs for Narrow Angle Camera (NAC) images
# Find more at https://wms.lroc.asu.edu/lroc/search
target_images = ["M119533355RE", "M114131109RE"]
# download_lro_subset(target_images)

**Model Training Logic**
****

In [ ]:
from torchmetrics import JaccardIndex, F1Score
from tqdm import tqdm

def train_one_epoch(model, dataloader, optimizer, criterion, device):
  model.train()
  total_loss = 0.0

  # Binary Metrics
  iou_metric = JaccardIndex(task="binary").to(device)
  f1_metic = F1Score(task="binary").to(device)

  pbar = tqdm(dataloader, desc="Training")
  for images, masks, prompts in pbar:

    # Move to GPU
    images = images.to(device)
    masks = masks.to(device)


    # Tokenize Prompts (CLIPTokenizer)
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)

    # Forward Pass
    optimizer.zero_grad()
    outputs = model(images, inputs['input_ids'], inputs['attention_mask'])

    # Calculate Loss
    loss = criterion(outputs, masks)


    # Backwards Pass
    loss.backward()
    optimizer.step()

    # Update Metrics
    preds = torch.sigmoid(outputs) > 0.5
    iou_metric.update(preds, masks.int())
    f1_metric.update(preds, masks.int())

    total_loss += loss.item()
    pbar.set_postfix({"Loss": loss.item(), "IoU": iou_metric.compute().item()})

  return total_loss / len(dataloader), iou_metric.compute(), f1_metric.compute()



**Generate Ground-Truth labels with LOLA DEM**
****
LOLA (Lunar Orbiter Laser Altimeter)

DEM (Digital Evaluation Model) data for each LRO image

In [ ]:
import numpy as np
import torch

def generate_lola_mask(dem_array, resolution_meters=118.0, slope_threshold=15.0):
  """
  dem_array: 2D numpy array of elevation values.
  resolution_meters: The spatial resolution of the DEM (m/pixel).
  slope_threshold: The degree at which terrain becomes 'hazardous'.
  """

  # Calculate gradient
  dx, dy = np.gradient(dem_array, resolution_meters)

  # Calculate slope magnitude
  slope = np.sqrt(dx**2 + dy**2)

  # Convert to degrees
  slope_deg = np.degrees(np.arctan(slope_magnitude))

  # Create binary mask: 1 for hazardous (Slope > Threshold), 0 for Safe
  mask = (slope_deg > slope_threshold).astype(np.uint8)

  return mask, slope_deg


**Baseline Vision-only UNet Model**
****

In [ ]:
import segmentation_models_pytorch as smp

def get_baseline_model():

  model = smp.Unet(
      encoder_name="resnet34",
      encoder_weights="imagenet",
      in_channels=3,
      classes=1
  )
  return model

**Vision-Language vs Vision Baseline UNet Model**
****